## Load packages

In [1]:
import Pkg
Pkg.activate("/home/yunanli/hpc2024/Github/NeuralCrop")

using NeuralCrop
using Lux, CUDA, LuxCUDA

import JLD2: @load, @save

CUDA.allowscalar(false)

const cdev = cpu_device()
const gdev = gpu_device()

  Activating project at `~/hpc2024/Github/NeuralCrop`


(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

## Initilization

In [2]:
@load "./initial_wheat.jld2" data

climate_path = "./"
pftparameters = cft1

### load neural network
model_file = "./neuralcrop_wheat.jld2"
@load model_file wheat_model wheat_ps wheat_st
ps, st = gdev(wheat_ps), gdev(wheat_st)

### time range: 2000-2019; load every 10-year climate data
all_indices = collect(1:length(data.coords))
batch_idx = all_indices;

## Simulations

In [3]:
function run_simulation!(
    n_decades,  
    model_type,         # "Hybrid" or "ProcessBased"
    pftparameters, 
    climbuf, 
    crop, 
    crop_cal, 
    photos, 
    pet, 
    soil, 
    managed_land,
    dailyWeather, 
    output,
    InitialData,
    climate_path,
    batch_idx,
    gdev;
    wheat_model = nothing,
    ps = nothing,        
    st = nothing,      
    start_day = 1,
    end_day = 3650    
)

    ### print
    println("\n" * "="^50)
    println("🚀 start simulation")
    println("="^50)

    for i in 1:n_decades

        start_year = 2000 + (i - 1) * 10
        end_year = start_year + 9

        print(" -> simulation years: $start_year - $end_year ... ")

        file_path = joinpath(climate_path, "climate_$(start_year)_$(end_year).jld2")
        @load file_path climate

        climate_loader = ClimateDataLoader(climate, batch_idx, gdev)
        
        if i == 1
            spin_up_climbuf!(pftparameters, climate_loader.temp_spinup, climbuf, 1, gdev)
        end

        data_batch = (
            latitude = InitialData.latitude, 
            climate = climate_loader, 
            lpjml = InitialData.lpjml
        )

        if model_type == "Hybrid"
            daily_crop_C3!(
                start_day, end_day, wheat_model, ps, st, 
                pftparameters, data_batch, length(batch_idx), 
                climbuf, crop, crop_cal, photos, pet, soil, 
                managed_land, dailyWeather, output, gdev
            )
        else
            daily_crop_C3!(
                start_day, end_day, 
                pftparameters, data_batch, length(batch_idx), 
                climbuf, crop, crop_cal, photos, pet, soil, 
                managed_land, dailyWeather, output, gdev
            )
        end

        println("completed ✅")
        
    end

    println("🎉 simulation completed!")
end

run_simulation! (generic function with 1 method)

## Hybrid simulations

In [4]:
### Initilization
InitialData = InitilDataLoader(data, batch_idx, gdev)
climbuf, crop, crop_cal, photos, pet, soil, managed_land, dailyWeather, output = init_structs!(cft1, InitialData, length(batch_idx), gdev);

### Running
run_simulation!(
    2, "Hybrid", pftparameters, climbuf, crop, crop_cal, 
    photos, pet, soil, managed_land, dailyWeather, output, 
    InitialData, climate_path, batch_idx, gdev;
    wheat_model = wheat_model, ps = ps, st = st
)


🚀 start simulation
 -> simulation years: 2000 - 2009 ... completed ✅
 -> simulation years: 2010 - 2019 ... completed ✅
🎉 simulation completed!


## Process-based simulations

In [5]:
### Initilization
InitialData = InitilDataLoader(data, batch_idx, gdev)
climbuf, crop, crop_cal, photos, pet, soil, managed_land, dailyWeather, output = init_structs!(cft1, InitialData, length(batch_idx), gdev);

### Running
run_simulation!(
    2, "ProcessBased", pftparameters, climbuf, crop, crop_cal, 
    photos, pet, soil, managed_land, dailyWeather, output, 
    InitialData, climate_path, batch_idx, gdev;
    wheat_model = wheat_model, ps = ps, st = st
)


🚀 start simulation
 -> simulation years: 2000 - 2009 ... completed ✅
 -> simulation years: 2010 - 2019 ... completed ✅
🎉 simulation completed!
